# Análise de Bandas Mais Relevantes - Dataset Real

Descobrir quais bandas ASTER são realmente importantes usando PCA.
As funções estão em `src/pixel_preprocessing.py`, importamos aqui apenas.


In [37]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '.')

# Importar funções do módulo pixel_preprocessing
from src.pixel_preprocessing import (
    prepare_pixel_data,
    standardize_bands,
    apply_pca,
    analyze_pca_loadings
)

# Carregar dataset
df = pd.read_csv(
    r'G:\.shortcut-targets-by-id\1JBfLtprp6bn1bXgQisabWu15-_qifNEN\ASTER_IMG\outputs\pixels_dataset.csv'
)
print(f"Dataset carregado: {df.shape}")
print(f"  Primeiras colunas: {df.columns.tolist()[:10]}")


OSError: [Errno 28] No space left on device

In [38]:
# Detectar número de bandas dinamicamente
pixel_cols = [col for col in df.columns if col.startswith('pixel_')]
if pixel_cols:
    max_band = max([int(col.split('_')[1]) for col in pixel_cols])
    num_bands = max_band + 1
else:
    num_bands = 14

band_names = [f'B{i+1:02d}' for i in range(num_bands)]

print("=" * 70)
print("PIPELINE: Prepare -> Standardize -> PCA -> Analyze")
print("=" * 70)

# Preparar dados de pixel
print("\n1. PREPARAR DADOS DE PIXEL")
df_pixels = prepare_pixel_data(df, band_names)
print(f"Shape: {df_pixels.shape}\n")

# Atualizar band_names com as bandas que realmente existem
band_names = [col for col in df_pixels.columns if col.startswith('B')]
print(f"Bandas encontradas: {band_names}\n")

# Padronizar bandas
print("2. PADRONIZAR BANDAS")
df_standardized, scaler = standardize_bands(df_pixels, band_names)
print(f"Shape: {df_standardized.shape}\n")

# Aplicar PCA
print("3. APLICAR PCA (variance_threshold=0.95)")
df_pca, pca_model = apply_pca(df_standardized, band_names, variance_threshold=0.95)
pc_columns = [f'PC{i+1}' for i in range(df_pca.shape[1])]
print(f"Shape: {df_pca.shape}")
print(f"Componentes: {pc_columns}\n")

# Analisar loadings e descobrir bandas importantes
print("4. ANALISAR LOADINGS - DESCOBRIR BANDAS IMPORTANTES")
df_loadings, important_bands = analyze_pca_loadings(pca_model, band_names, pc_columns)

Número de colunas de pixel (147456) não corresponde ao esperado (2415919104). Processando ainda assim...


PIPELINE: Prepare -> Standardize -> PCA -> Analyze

1. PREPARAR DADOS DE PIXEL


MemoryError: Unable to allocate 332. MiB for an array with shape (147464, 295) and data type object

In [9]:
# Pipeline rodado nas celulas anteriores
print("Pipeline executado com sucesso!")

1. PREPARAR DADOS DE PIXEL
Preparando os dados de pixel...
Pixels por chip esperado (height x width): 16384
Colunas de pixel encontradas: 147456
Número de bandas detectado: 9
⚠️  Detectado 9 bandas, mas band_names tem 14
   Usando as primeiras 9 bandas
Convertendo colunas de banda para tipo numérico...
✓ Reestruturação completa.
Shape de df_pixels: (4833280, 10)

Bandas detectadas: ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09']

2. PADRONIZAR BANDAS
Aplicando StandardScaler às colunas de banda...
✓ Padronização completa.
Shape padronizado: (4833280, 10)

3. APLICAR PCA
Realizando PCA para determinar o número ideal de componentes (variância > 95.0%)...
✓ Número de componentes: 2
✓ Transformação PCA completa.
Shape PCA: (4833280, 2)

4. ANALISAR LOADINGS E BANDAS IMPORTANTES
Analisando loadings dos componentes principais...

Loadings dos Componentes Principais:
          B01       B02       B03       B04       B05       B06       B07  \
PC1  0.316939  0.308801  0.306812 

In [39]:
print("\n" + "=" * 70)
print("RESULTADO: BANDAS MAIS IMPORTANTES POR COMPONENTE")
print("=" * 70)

for pc, bands_list in important_bands.items():
    var_explained = pca_model.explained_variance_ratio_[int(pc.replace('PC', ''))-1] * 100
    print(f"\n{pc} ({var_explained:.2f}% da variância):")
    print("-" * 50)
    for rank, (band, loading) in enumerate(bands_list[:7], 1):
        print(f"  {rank}. {band} | loading={loading:+.4f}")

print("\n" + "=" * 70)
print("RANKING GLOBAL: TOP BANDAS EM QUALQUER COMPONENTE")
print("=" * 70)

all_bands_importance = {}
for pc, bands_list in important_bands.items():
    for band, loading in bands_list[:5]:
        if band not in all_bands_importance:
            all_bands_importance[band] = []
        all_bands_importance[band].append(abs(loading))

band_avg = {band: np.mean(vals) for band, vals in all_bands_importance.items()}
sorted_final = sorted(band_avg.items(), key=lambda x: x[1], reverse=True)

for rank, (band, avg_importance) in enumerate(sorted_final[:7], 1):
    print(f"{rank}. {band} | média = {avg_importance:.4f}")

print("\nAnálise concluída!")


RESULTADO: BANDAS MAIS IMPORTANTES POR COMPONENTE

PC1 (90.60% da variância):
--------------------------------------------------
  1. B08 | loading=+0.3474
  2. B07 | loading=+0.3468
  3. B06 | loading=+0.3455
  4. B09 | loading=+0.3451
  5. B05 | loading=+0.3420
  6. B04 | loading=+0.3372
  7. B01 | loading=+0.3169

PC2 (5.63% da variância):
--------------------------------------------------
  1. B02 | loading=+0.6343
  2. B01 | loading=+0.5819
  3. B04 | loading=-0.3448
  4. B05 | loading=-0.2534
  5. B06 | loading=-0.1932
  6. B07 | loading=-0.1488
  7. B09 | loading=-0.1061

RANKING GLOBAL: TOP BANDAS EM QUALQUER COMPONENTE
1. B02 | média = 0.6343
2. B01 | média = 0.5819
3. B08 | média = 0.3474
4. B07 | média = 0.3468
5. B09 | média = 0.3451
6. B04 | média = 0.3448
7. B05 | média = 0.2977

Análise concluída!


In [40]:
print("\n" + "=" * 70)
print("PREPARAR DADOS PARA REDE NEURAL")
print("=" * 70)

# Recarregar módulo
import importlib
import src.pixel_preprocessing
importlib.reload(src.pixel_preprocessing)

from src.pixel_preprocessing import prepare_for_neural_network

# Preparar datasets train/test
nn_dataset = prepare_for_neural_network(df_pca, test_size=0.2, random_state=42)

print(f"\nDatasets preparados:")
print(f"  - X_train shape: {nn_dataset['X_train'].shape}")
print(f"  - X_test shape: {nn_dataset['X_test'].shape}")
print(f"  - Features (PCs): {nn_dataset['feature_names']}")
print(f"  - Input shape para rede: (batch_size, {nn_dataset['n_features']})")

print("\nExemplo de uso em Keras/TensorFlow:")
print("""
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

X_train = nn_dataset['X_train']
X_test = nn_dataset['X_test']

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50)
""")


PREPARAR DADOS PARA REDE NEURAL

Datasets preparados:
  - X_train shape: (3866624, 2)
  - X_test shape: (966656, 2)
  - Features (PCs): ['PC1', 'PC2']
  - Input shape para rede: (batch_size, 2)

Exemplo de uso em Keras/TensorFlow:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

X_train = nn_dataset['X_train']
X_test = nn_dataset['X_test']

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50)

